In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install sentence-transformers
!pip install faiss-gpu-cu12

In [ ]:
import torch
import json
import pandas as pd
import numpy as np
import faiss
import networkx as nx
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import CrossEncoder

In [ ]:
CORPUS_PATH = "/kaggle/input/datasets/minhvb10/zalo-dataset/corpus.csv"
TEST_PATH   = "/kaggle/input/datasets/minhvb10/zalo-dataset/crawled_test_5k.jsonl"
MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
EMBED_BATCH = 512
K_GRAPH = 10
TOP_K_RETRIEVAL = 100
RECALL_KS = [1, 5, 10]

In [ ]:
def load_corpus_csv(path):
    df = pd.read_csv(path)
    df["law_id"] = df["law_id"].str.lower().str.replace("đ", "d")
    docs = df.to_dict(orient="records")
    return docs

In [ ]:
def load_test_jsonl(path):
    tests = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj.get("question", "").strip()
            rels = obj.get("relevant_articles", [])
            
            if q and rels:
                tests.append({
                    "query": q, 
                    "relevant_articles": rels  
                })
    return tests

In [ ]:
def embed(model, texts, batch_size=EMBED_BATCH):
    all_embs = []
    all_embs = model.encode(
        texts, 
        batch_size=batch_size, 
        convert_to_numpy=True, 
        show_progress_bar=True 
    )
    return all_embs.astype(np.float32)

In [ ]:
import re

def extract_referenced_law_ids(text):
    """
    Hàm sử dụng Regex để tìm các law_id được nhắc đến trong text (title).
    Pattern bắt các dạng: 120/2011/nđ-cp, 57/nq-cp, 01/2002/qh11...
    """
    if not isinstance(text, str):
        return []
    
    # Regex pattern: 
    # \d+ : số hiệu
    # / : dấu gạch chéo
    # [a-zA-Z0-9\-\.]+ : phần ký hiệu (vd: nđ-cp, qh14)
    # Pattern này tìm chuỗi dạng: số/năm/ký-hiệu
    pattern = r"(\d+/\d+/[a-zA-Z0-9\-\đ]+)"
    
    matches = re.findall(pattern, text.lower())
    
    # Chuẩn hóa giống như cách xử lý law_id đầu vào
    normalized_ids = [m.replace("đ", "d") for m in matches]
    return normalized_ids

def build_hierarchical_graph(docs, embeddings, k):
    """
    Builds hierarchical graph: Field -> Law -> Article -> Khoan -> Diem
    + RELATIONSHIP: Law references Law based on Title
    """
    G = nx.Graph()
    seen_laws = set()
    seen_articles = set()
    seen_khoans = set()
    
    # Dictionary để lưu danh sách các reference tìm thấy: {source_law_id: [target_law_ids]}
    law_references_map = {} 
    
    print("Building hierarchical graph (Law -> Article -> Khoan -> Diem)...")
    
    for i, doc in enumerate(tqdm(docs, desc="Building hierarchy")):
        
        # 1. Create the Diem (text) Node
        diem_node_id = str(doc["id"])
        G.add_node(
            diem_node_id, type="Diem", title=doc.get("title", ""), text=doc.get("text", ""),
            law_id=doc.get("law_id"), article_id=doc.get("article_id"), 
            khoan_id=doc.get("khoan_id"), diem_id=doc.get("diem_id")
        )
        
        # 2. Create and link Khoan node
        if doc.get("khoan_id") and not pd.isna(doc.get("khoan_id")):
            khoan_node_id = f"{doc['law_id']}_{doc['article_id']}_{doc['khoan_id']}"
            if khoan_node_id not in seen_khoans:
                G.add_node(khoan_node_id, type="Khoan", label=f"Khoản {doc['khoan_id']}")
                seen_khoans.add(khoan_node_id)
            G.add_edge(khoan_node_id, diem_node_id, relation="CONTAINS")
            
            # 3. Create and link Article node
            if doc.get("article_id") and not pd.isna(doc.get("article_id")):
                article_node_id = f"{doc['law_id']}_{doc['article_id']}"
                if article_node_id not in seen_articles:
                    G.add_node(article_node_id, type="Article", label=f"Điều {doc['article_id']}")
                    seen_articles.add(article_node_id)
                G.add_edge(article_node_id, khoan_node_id, relation="CONTAINS")
                
                # 4. Create and link Law node
                if doc.get("law_id") and not pd.isna(doc.get("law_id")):
                    law_node_id = str(doc["law_id"])
                    
                    if law_node_id not in seen_laws:
                        G.add_node(law_node_id, type="Law", label=f"Law {doc['law_id']}")
                        seen_laws.add(law_node_id)
                        
                        # Chỉ xử lý 1 lần cho mỗi Law ID để tránh lặp lại
                        doc_title = doc.get("title", "")
                        refs = extract_referenced_law_ids(doc_title)
                        if refs:
                            # Lọc bỏ reference đến chính nó (nếu có)
                            refs = [r for r in refs if r != law_node_id]
                            if refs:
                                law_references_map[law_node_id] = refs
                        # --------------------------------------------------
                    
                    G.add_edge(law_node_id, article_node_id, relation="CONTAINS")
    
    print(f"Created {len(seen_laws)} Law nodes")
    print(f"Created {len(seen_articles)} Article nodes")
    print(f"Created {len(seen_khoans)} Khoan nodes")
    #------------------------------------------------------------------
    print("\nAdding REFERENCES edges between Laws based on titles...")
    ref_edges_count = 0
    for source_law, targets in law_references_map.items():
        for target_law in targets:
            # (Tránh tạo node rác không có nội dung)
            if target_law in seen_laws:
                if not G.has_edge(source_law, target_law):
                    G.add_edge(
                        source_law,
                        target_law,
                        relation="REFERENCES",
                        weight=0.5  
                    )
                    ref_edges_count += 1
    
    print(f"Added {ref_edges_count} REFERENCES edges (Law <-> Law)")
    # -------------------------------------------------------------
    vectors = embeddings.copy()
    faiss.normalize_L2(vectors)
    global_index = faiss.IndexFlatIP(vectors.shape[1])
    global_index.add(vectors)
    
    print(f"\nAdding SIMILAR_TO edges (k={k})...")
    similar_edges_count = 0
    
    D, I = global_index.search(vectors, k + 1)
    
    for i, doc in enumerate(tqdm(docs, desc="Adding similarity edges")):
        node_id_i = str(doc["id"])
        for local_j, sim in zip(I[i][1:], D[i][1:]): 
            global_j = local_j
            node_id_j = str(docs[global_j]["id"])
            if not G.has_edge(node_id_i, node_id_j):
                G.add_edge(node_id_i, node_id_j, relation="SIMILAR_TO", weight=float(sim))
                similar_edges_count += 1
                
    print(f"Added {similar_edges_count} SIMILAR_TO edges")
    
    return G

In [ ]:
def build_faiss(embs):
    d = embs.shape[1]
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(d)
    index.add(embs)
    return index

In [ ]:
def retrieve_and_rerank_with_graph(index, G, corpus_ids, q_emb_single, top_n_vector):
    """
    BEFORE CROSS-ENCODER: Vector Search -> Graph Expansion (Max Pooling)
    """
    D, I = index.search(np.array([q_emb_single]), top_n_vector)
    
    node_scores = {}
    for i, idx in enumerate(I[0]):
        if idx != -1:
            doc_id = corpus_ids[idx]
            node_scores[doc_id] = D[0][i] 
            
    final_scores = node_scores.copy()

    RELATION_WEIGHTS = {
        "SIMILAR_TO": 0.85,   
        "CONTAINS": 0.85,     
        "REFERENCES": 0.95,   
        "DEFAULT": 0.1
    }

    neighbor_boosts = {}

    for doc_id, v_score in node_scores.items():
        if doc_id in G.nodes():
            neighbors = list(G.neighbors(doc_id))
            for neighbor in neighbors:
                edge_data = G.get_edge_data(doc_id, neighbor)
                relation_type = edge_data.get("relation", "DEFAULT") 
                current_weight = RELATION_WEIGHTS.get(relation_type, RELATION_WEIGHTS["DEFAULT"])
                boost_score = v_score * current_weight

                # Max Pooling to prevent noise amplification
                if neighbor not in neighbor_boosts:
                    neighbor_boosts[neighbor] = boost_score
                else:
                    neighbor_boosts[neighbor] = max(neighbor_boosts[neighbor], boost_score)

    for neighbor, max_boost in neighbor_boosts.items():
        if neighbor not in final_scores:
            final_scores[neighbor] = max_boost
        else:
            final_scores[neighbor] = max(final_scores[neighbor], max_boost)

    ranked_docs = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in ranked_docs]


def retrieve_graph_and_cross_encoder(index, G, corpus_ids, doc_text_map, q_emb_single, query_text, reranker, top_n_vector=100, top_k_rerank=30):
    """
    AFTER CROSS-ENCODER: Vector -> Graph Expansion -> Top K -> Cross-Encoder
    """
    #Get the candidates from the Graph
    graph_ranked_ids = retrieve_and_rerank_with_graph(index, G, corpus_ids, q_emb_single, top_n_vector)
    
    candidate_ids = graph_ranked_ids[:top_k_rerank]
    
    #Prepare pairs for the Cross-Encoder: [Query, Document_Text]
    cross_inputs = []
    valid_ids = []
    
    for doc_id in candidate_ids:
        if doc_id in doc_text_map:
            doc_text = doc_text_map[doc_id]
            cross_inputs.append([query_text, doc_text])
            valid_ids.append(doc_id)
            
    if not cross_inputs:
        return candidate_ids # Fallback if empty
        
    # Predict scores
    cross_scores = reranker.predict(cross_inputs, batch_size=256)
    
    # Sort based on the new Cross-Encoder scores
    reranked_results = sorted(zip(valid_ids, cross_scores), key=lambda x: x[1], reverse=True)
    
    return [doc_id for doc_id, score in reranked_results]

def get_ground_truth_ids(relevant_articles):
    """
    Trích xuất danh sách các ID từ ground_truths trong file JSONL
    Giả sử ID của corpus tương ứng với ans_id
    """
    gt_ids = []
    for article in relevant_articles:
        if "ans_id" in article:
            gt_ids.extend([str(ans_id) for ans_id in article["ans_id"]])
    return gt_ids

def run_comparative_evaluation(index, G, corpus_ids, doc_text_map, queries, q_embs, ground_truths, reranker):
    k_list = [1, 5, 10]
    num_queries = len(q_embs)
    
    # Lấy kết quả từ Graph cho queries
    all_graph_results = []
    print("Step 1: Running Graph Retrieval for all queries...")
    for i in tqdm(range(num_queries), desc="Graph Retrieval"):
        ranked_graph = retrieve_and_rerank_with_graph(
            index, G, corpus_ids, q_embs[i], top_n_vector=100
        )
        all_graph_results.append(ranked_graph)
        
    # Chuẩn bị dữ liệu cho Cross-Encoder 
    print("Step 2: Preparing Cross-Encoder inputs...")
    top_k_rerank = 30
    all_cross_inputs = []
    query_to_input_indices = [] # Lưu vị trí của từng query trong list tổng
    
    current_idx = 0
    for i in range(num_queries):
        candidate_ids = all_graph_results[i][:top_k_rerank]
        valid_ids_for_query = []
        
        for doc_id in candidate_ids:
            if doc_id in doc_text_map:
                doc_text = doc_text_map[doc_id]
                all_cross_inputs.append([queries[i], doc_text])
                valid_ids_for_query.append(doc_id)
                
        # Lưu lại khoảng index của các candidates thuộc về query i
        num_candidates = len(valid_ids_for_query)
        query_to_input_indices.append((current_idx, current_idx + num_candidates, valid_ids_for_query))
        current_idx += num_candidates

    print(f"Step 3: Reranking {len(all_cross_inputs)} pairs using Cross-Encoder...")
    all_cross_scores = []
    if all_cross_inputs:
        all_cross_scores = reranker.predict(all_cross_inputs, batch_size=256)

    # Đánh giá Metrics
    print("Step 4: Calculating Metrics...")
    hits_graph = {k: 0 for k in k_list}
    recalls_graph = {k: 0.0 for k in k_list}
    hits_ce = {k: 0 for k in k_list}
    recalls_ce = {k: 0.0 for k in k_list}
    
    for i in range(num_queries):
        gt_ids = set(get_ground_truth_ids(ground_truths[i]))
        if not gt_ids:
            continue
            
        # Graph Metrics
        ranked_graph = all_graph_results[i]
        for k in k_list:
            preds_graph = set(ranked_graph[:k])
            correct_graph = preds_graph.intersection(gt_ids)
            if len(correct_graph) > 0: hits_graph[k] += 1
            recalls_graph[k] += len(correct_graph) / len(gt_ids)
            
        # Trích xuất kết quả CE cho query i
        start_idx, end_idx, valid_ids = query_to_input_indices[i]
        if start_idx < end_idx: # Có candidates
            scores = all_cross_scores[start_idx:end_idx]
            reranked_results = sorted(zip(valid_ids, scores), key=lambda x: x[1], reverse=True)
            ranked_ce = [doc_id for doc_id, score in reranked_results]
        else:
            ranked_ce = ranked_graph[:top_k_rerank] # Fallback
            
        # Cross-Encoder Metrics
        for k in k_list:
            preds_ce = set(ranked_ce[:k])
            correct_ce = preds_ce.intersection(gt_ids)
            if len(correct_ce) > 0: hits_ce[k] += 1
            recalls_ce[k] += len(correct_ce) / len(gt_ids)

    print("\n" + "="*60)
    print("1. BEFORE CROSS-ENCODER (GRAPHRAG)")
    print("="*60)
    for k in k_list:
        print(f"  Recall@{k:2d}: {recalls_graph[k]/num_queries:.4f}  |  Hit@{k:2d}: {hits_graph[k]/num_queries:.4f}")

    print("\n" + "="*60)
    print("2. AFTER CROSS-ENCODER (GRAPHRAG + RERANKER)")
    print("="*60)
    for k in k_list:
        print(f"  Recall@{k:2d}: {recalls_ce[k]/num_queries:.4f}  |  Hit@{k:2d}: {hits_ce[k]/num_queries:.4f}")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

print("\nLoading corpus CSV...")
corpus = load_corpus_csv(CORPUS_PATH)
print(f"Corpus size: {len(corpus):,}")

print("Creating document text map...")
doc_text_map = {str(d["id"]): f"{d.get('title', '')} {d.get('text', '')}" for d in corpus}

corpus_texts = [f"{d['title']} {d['text']}" for d in corpus]
corpus_ids = [str(d["id"]) for d in corpus]

# Create map id -> index
id2idx = {id_: i for i, id_ in enumerate(corpus_ids)}

print("\nLoading test set...")
tests = load_test_jsonl(TEST_PATH)
print(f"Test queries: {len(tests):,}")

queries = [t["query"] for t in tests]
ground_truths = [t["relevant_articles"] for t in tests]

print("\nLoading embedding model...")
model = SentenceTransformer(MODEL_NAME, device=device)

if device == 'cuda': model.half()

print("\nEmbedding corpus...")
corpus_emb = embed(model, corpus_texts)
print(f"Embedding shape: {corpus_emb.shape}")

print("\nBuilding FAISS index...")
index = build_faiss(corpus_emb)

print("\n" + "="*70)
print("BUILDING HIERARCHICAL GRAPH (NO FIELD)")
print("="*70)

G = build_hierarchical_graph(corpus, corpus_emb, k=K_GRAPH)

print(f"\n{'='*70}")
print("GRAPH STATISTICS")
print("="*70)
print(f"Total nodes: {G.number_of_nodes():,}")
print(f"Total edges: {G.number_of_edges():,}")

# Count nodes by type
node_types = {}
for node in G.nodes():
    node_type = G.nodes[node].get('type', 'Unknown')
    node_types[node_type] = node_types.get(node_type, 0) + 1

print("\nNode type distribution:")
# Removed 'Field' from this list
for node_type in ['Law', 'Article', 'Khoan', 'Diem']: 
    count = node_types.get(node_type, 0)
    print(f"  {node_type:15s}: {count:,} nodes")

# Count edges by type
edge_types = {}
for u, v, data in G.edges(data=True):
    edge_type = data.get('relation', 'Unknown')
    edge_types[edge_type] = edge_types.get(edge_type, 0) + 1

print("\nEdge type distribution:")
for edge_type, count in edge_types.items():
    print(f"  {edge_type:15s}: {count:,} edges")

print("\n" + "="*70)
print("EMBEDDING QUERIES")
print("="*70)
q_emb = embed(model, queries)
print(f"Query embeddings shape: {q_emb.shape}")

print("Loading Cross-Encoder model...")
reranker = CrossEncoder(RERANK_MODEL, max_length=512, device=device)
if device == 'cuda': reranker.model.half()
run_comparative_evaluation(index, G, corpus_ids, doc_text_map, queries, q_emb, ground_truths, reranker)

print("\n" + "="*70)
print("EVALUATION COMPLETE!")
print("="*70)